In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.customer_health_base AS (
WITH leads_processed_deduped AS (
    SELECT
    *
    FROM (
        SELECT
            id AS lead_id,
            status_label,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed
        WHERE
            status_label = 'Coaching Client'
        ) 
        WHERE
            rn = 1
)
,
leads_processed_custom_deduped AS (
    SELECT
        *
    FROM (
        SELECT
            leads_processed_id AS lead_id,
            CSM AS csm_id, -- — customer success manager
            Contracted_Value, -- — sales context
            Tier, -- — account info
            Avatar, -- profile image
            bronze_insert_date,
            ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
    )
    WHERE
        rn = 1
)
SELECT
    l.lead_id,
    l.status_label,
    l.date_updated,
    l.bronze_insert_date,
    NULLIF(c.csm_id, '\\N') AS csm_id,
    NULLIF(c.Contracted_Value, '\\N') AS Contracted_Value,
    NULLIF(c.Tier, '\\N') AS Tier,
    NULLIF(c.Avatar, '\\N') AS Avatar
FROM
    leads_processed_deduped l
LEFT JOIN
    leads_processed_custom_deduped c
ON
    l.lead_id = c.lead_id
)


In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_email AS (
WITH deduped_id_activities_filtered_email AS (
    SELECT
        a.id AS acitvity_id,
        a.lead_id, 
        a.type,
        a.status,
        a.activity_at,
        b.body_text,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed a
    LEFT JOIN
        crm_ingestion.silver.lead_activites_email_body b
    ON
        a.id = b.activity_id
    WHERE 
        a.type = 'Email'
    AND a.status = 'inbox'
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%A new event has been scheduled.%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%is requesting access to the following folder%')
    AND (b.body_text IS NULL OR b.body_text NOT ILIKE '%requests access to an item%')
),
latest_email_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_email
    WHERE
        id_rn = 1
)
    SELECT
        lead_id,
        DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_email
    FROM
        latest_email_per_lead_id
    WHERE
        lead_id_rn = 1
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting AS (
WITH deduped_id_activities_filtered_text AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_email_body
    WHERE
        body_text ILIKE '%A new event has been scheduled.%'
),
latest_text_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_text
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_meeting
FROM
    latest_text_per_lead_id
WHERE
    lead_id_rn = 1  
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.missed_calls_14_days AS (  
WITH deduped_id AS (
    SELECT
        lead_id,
        id AS activity_id,
        status,
        activity_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Call'
        AND direction = 'inbound'
        AND disposition = 'no-answer'
),
deduped_calls AS (
    SELECT *
    FROM deduped_id
    WHERE id_rn = 1
)
SELECT
    lead_id,
    COUNT(*) AS missed_calls_14_days
FROM
    deduped_calls
WHERE
    DATEDIFF(CURRENT_DATE, DATE(activity_at)) <= 14
GROUP BY
    lead_id
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_until_next_meeting AS (
WITH deduped_id AS (
    SELECT
        id AS activity_id,
        lead_id,
        status,
        starts_at,
        date_updated,
        bronze_insert_date,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Meeting'
        AND status = 'upcoming'
        AND DATE(starts_at) > CURRENT_DATE
),
earliest_upcoming_per_lead AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY starts_at ASC) AS lead_id_rn
    FROM
        deduped_id
    WHERE
        id_rn = 1
)
SELECT
    lead_id,
    DATEDIFF(DATE(starts_at), CURRENT_DATE) AS days_until_next_meeting
FROM
    earliest_upcoming_per_lead
WHERE
    lead_id_rn = 1
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_login AS (
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
crm_to_moodle AS (
    SELECT DISTINCT
        c.leads_processed_id AS lead_id,
        m.LASTACCESS
    FROM crm_ingestion.silver.leads_processed_contacts c
    JOIN crm_ingestion.silver.mdl_users_processed m
      ON c.name = m.USERNAME
    WHERE m.LASTACCESS IS NOT NULL
      AND m.LASTACCESS != ''
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.leads_processed_id ORDER BY m.LASTACCESS DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(m.LASTACCESS)) AS days_since_last_login
FROM coaching_clients cc
JOIN crm_to_moodle m ON cc.lead_id = m.lead_id
)

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting_calendly AS (   
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_events AS (
    SELECT
        INVITEE_NAME,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE INVITEE_NAME IS NOT NULL
      AND EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) < CURRENT_DATE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC) = 1
),
contacts_bridge AS (
    SELECT DISTINCT
        lc.name,
        lc.leads_processed_id AS lead_id
    FROM crm_ingestion.silver.leads_processed_contacts lc
    WHERE lc.name IS NOT NULL
),
calendly_matched AS (
    SELECT
        cb.lead_id,
        ce.EVENT_START_TIME
    FROM contacts_bridge cb
    JOIN calendly_events ce ON ce.INVITEE_NAME = cb.name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY cb.lead_id ORDER BY ce.EVENT_START_TIME DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(cm.EVENT_START_TIME)) AS days_since_last_meeting_calendly
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
)

In [0]:
%sql
-- Sentiment + message recency per lead_id
-- Data originates from student_sentiment source, already joined to leads by CRM
-- and stored in leads_processed_custom. clean \N → NULL and exclude C082U471YMU.
CREATE OR REPLACE TABLE crm_ingestion.gold.sentiment AS (
SELECT
    leads_processed_id AS lead_id,
    NULLIF(SENTIMENTS_LAST_30_DAYS, '\\N') AS sentiment,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT AS BIGINT) AS days_since_last_message_from_client,
    TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER AS BIGINT) AS days_since_last_message_from_team_member
FROM (
    SELECT
        leads_processed_id,
        Channel_ID,
        SENTIMENTS_LAST_30_DAYS,
        DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
        DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
        ROW_NUMBER() OVER(PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
    FROM
        crm_ingestion.silver.leads_processed_custom
    WHERE
        Channel_ID IS NOT NULL
        AND Channel_ID != ''
        AND Channel_ID != 'C082U471YMU'  -- excluded channel (spec 5.1)
)
WHERE rn = 1
)